In [ ]:
import yaml
import json
from app.datasets.loader import load_multiple_test_cases
from app.datasets.validator import validate_dataset_schema
from app.datasets.preprocessing import preprocess_dataframe
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests

In [ ]:

file_list = [
  './app/data/raw/tramites.xlsx',
  './app/data/raw/accesibilidad.xlsx',
  './app/data/raw/descubrir.xlsx',
  './app/data/raw/solicitudes.xlsx',
  './app/data/raw/organigrama.xlsx'
]

test_config = {
  'TIMINGS': True,
  'TOKENS': True,
  'FOUNDRYS': True,
  'TRIAGE': True,
  'ROUTER': True,
  'GROUNDING': True,
}   

df = load_multiple_test_cases(file_list)
df = validate_dataset_schema(df)
# df = preprocess_dataframe(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)  

In [ ]:
responses = client.query_batch(df['user_input'],df['reference'])

In [ ]:
save_responses_in_json, response_file_path = client.save_api_responses(responses)

# response_file_path = './app/data/processed/outcome_baseline_20260204-125139.json'

In [ ]:
with open(response_file_path, 'r', encoding='UTF-8') as f:
  data = json.load(f)

results = run_tests(
  config = test_config, 
  data = data, 
  df = df, 
  timestamp = str(response_file_path)
)

In [ ]:
import os
from azure.cosmos import CosmosClient
from dotenv import load_dotenv

load_dotenv()

ENDPOINT = os.getenv('COSMOS_ENDPOINT')
KEY = os.getenv('COSMOS_KEY')
DATABASE = os.getenv('DATABASE_NAME')
CONTAINER = 'datasources'


client = CosmosClient(ENDPOINT, credential=KEY)
database = client.get_database_client(DATABASE)
container = database.get_container_client(CONTAINER)



In [ ]:
import json
with open('./app/data/processed/results_20260205-120729.json', 'r', encoding='UTF-8') as f:
  data = json.load(f)

data['id'] = data.get('timestamp')

response = container.create_item(body=data)

print("Item created:", response)

INFO:azure.cosmos._cosmos_http_logging_policy:Request URL: 'https://cosmosdb-shared-dev.documents.azure.com:443/'
Request method: 'GET'
Request headers:
    'Cache-Control': 'no-cache'
    'x-ms-version': '2020-07-15'
    'x-ms-documentdb-query-iscontinuationexpected': 'False'
    'x-ms-consistency-level': 'Session'
    'x-ms-cosmos-sdk-supportedcapabilities': '1'
    'x-ms-activity-id': 'b468d812-f54e-4f93-88bb-de273402dc5c'
    'x-ms-date': 'Mon, 09 Feb 2026 01:36:16 GMT'
    'authorization': 'REDACTED'
    'Accept': 'application/json'
    'x-ms-client-id': '230e0c25-9e17-4e36-ba00-ab794d5df1c4'
    'x-ms-thinclient-proxy-resource-type': 'databaseaccount'
    'x-ms-thinclient-proxy-operation-type': 'Read'
    'Content-Length': '0'
    'User-Agent': 'azsdk-python-cosmos/4.14.6 Python/3.12.3 (Windows-11-10.0.26100-SP0)'
No body was attached to the request
INFO:azure.cosmos._cosmos_http_logging_policy:Request URL: 'https://cosmosdb-shared-dev-eastus2.documents.azure.com:443/dbs/db-agent

Item created: {'timestamp': '20260205-120729', 'nodes': {'triage': {'positives': 802, 'total': 884, 'result': 90.72}, 'router': {'positives': 753, 'total': 802, 'result': 93.89}, 'grounding': {'positives': 404, 'total': 412, 'result': 98.06}}, 'timings': {'reformulate': {'prom': 1.284, 'p90': 1.844, 'p95': 2.171, 'quantity': 879}, 'triage': {'prom': 1.62, 'p90': 2.363, 'p95': 2.967, 'quantity': 879}, 'router': {'prom': 1.488, 'p90': 2.139, 'p95': 2.606, 'quantity': 879}, 'ag_call': {'prom': 4.294, 'p90': 6.227, 'p95': 7.611, 'quantity': 412}, 'personality': {'prom': 3.502, 'p90': 4.899, 'p95': 5.669, 'quantity': 412}, 'grounding': {'prom': 2.232, 'p90': 3.122, 'p95': 3.701, 'quantity': 412}, 'retriever': {'prom': 0.602, 'p90': 0.661, 'p95': 1.516, 'quantity': 412}, 'ret_embeddings': {'prom': 0.278, 'p90': 0.337, 'p95': 1.167, 'quantity': 412}, 'rag_answer': {'prom': 2.741, 'p90': 4.127, 'p95': 4.772, 'quantity': 412}, 'response_time': {'prom': 8.612, 'p90': 12.332, 'p95': 13.868, 'quan

INFO:azure.cosmos._cosmos_http_logging_policy:Response status: 200
Response headers:
    'Content-Length': '1800'
    'Date': 'Mon, 09 Feb 2026 01:36:23 GMT'
    'Content-Type': 'application/json'
    'Server': 'Microsoft-HTTPAPI/2.0'
    'x-ms-gatewayversion': 'version=2.14.0'
    'Cache-Control': 'no-store, no-cache'
    'Pragma': 'no-cache'
    'x-ms-max-media-storage-usage-mb': '0'
    'x-ms-media-storage-usage-mb': '0'
    'x-ms-databaseaccount-consumed-mb': 'REDACTED'
    'x-ms-databaseaccount-reserved-mb': 'REDACTED'
    'x-ms-databaseaccount-provisioned-mb': 'REDACTED'
    'Strict-Transport-Security': 'max-age=31536000'
    'Content-Location': 'https://cosmosdb-shared-dev.documents.azure.com/'
INFO:azure.cosmos._cosmos_http_logging_policy:Request URL: 'https://cosmosdb-shared-dev-eastus2.documents.azure.com:443/'
Request method: 'GET'
Request headers:
    'Cache-Control': 'no-cache'
    'x-ms-version': '2020-07-15'
    'x-ms-documentdb-query-iscontinuationexpected': 'False'
   